# 水力発電候補地選定システム

## このシステムでできること

地名を入力するだけで、その地域で**水力発電に適した場所**を自動で探し出します。

- **地図上に候補地を表示**（水源・取水口・発電所の位置）
- **発電量を自動計算**（どのくらいの電力が作れるか）
- **グラフで比較**（複数の候補地を見やすく比較）
- **結果をファイル保存**（CSV・HTML・PNG形式）

## 使い方（3ステップ）

### ステップ1：地名を入力
```python
selector = HydroSiteSelector("松本市")
```
※ 日本国内の市区町村名を入力してください

### ステップ2：分析を実行
```python
map_result, fig_result = selector.run_analysis()
```
※ 実行時間の目安：数分～十数分程度（地域の広さによって変わります）

### ステップ3：結果を確認
- **地図**: 上位3組の候補地が色分けされて表示されます
  - 赤色：第1位の組合せ
  - 青色：第2位の組合せ
  - 緑色：第3位の組合せ
- **グラフ**: 標高分布や発電量の比較が表示されます
- **データ**: CSV形式で詳細データが保存されます

## システムの仕組み

### 探索範囲の決定方法

1. **正確な行政区画境界を取得**
   - OpenStreetMapから行政区画の境界ポリゴンを取得
   - 数万点の頂点データで正確な範囲を定義
   - 面積誤差は通常1%未満の高精度

2. **境界内のみを分析**
   - グリッドポイントを境界内にフィルタリング
   - 河川データも境界と交差するもののみを使用
   - 隣接自治体のデータは除外

### 候補地の選定方法（詳細）

システムは3段階で候補地を絞り込んでいきます：

#### 第1段階：グリッドポイントの生成と評価

1. **グリッドの生成**
   - 探索範囲を格子状に分割（例：28×28 = 784点）
   - 行政区画の境界内のポイントのみを残す（フィルタリング）
   - 実際の候補点数は境界形状によって変動（通常30～60%が境界内）

2. **標高データの取得**
   - 各グリッドポイントの標高をOpen-Elevation APIで取得
   - バッチ処理で高速化（150点ずつまとめて取得）
   - 標高範囲の把握と地形の理解

3. **地形勾配の計算**
   - 各ポイントの周囲8方向の標高差を計算
   - 勾配の大きさ：地形の急峻さを示す
   - 勾配の向き：水の流れる方向を示す

#### 第2段階：各施設タイプの候補地選定

**1. 水源候補地の選定**

評価基準（スコアが高いほど良い）：

```
水源スコア = (標高の高さ) × 1.0 + (推定流量) × 0.3 + (勾配の大きさ) × 0.2
```

- **標高の重視**: 高い場所ほど大きな落差が得られる
- **流量の考慮**: 近くの河川から推定した流量を加点
- **勾配の活用**: 適度な勾配は水が集まりやすいことを示す
- **選定数**: 指定した数（デフォルト40箇所）の上位候補を選定

**河川流量の推定方法**:
1. 各ポイントから最も近い河川を探索（通常5km以内）
2. 河川タイプによる基準流量
   - `river`（大河川）: 1.0 m³/s
   - `stream`（小河川）: 0.5 m³/s
3. 距離による減衰計算
   - 減衰率 = exp(-距離 / 2.0)
   - 遠い河川ほど影響が小さくなる

**2. 取水口候補地の選定**

評価基準：

```
取水口スコア = (標高の中程度さ) × 1.0 + (勾配の緩やかさ) × 0.5
```

- **標高の中間性**: 水源より低く、発電所より高い中間標高を優先
- **勾配の緩やかさ**: 建設しやすい平坦な場所を優先
  - 逆数スコアを使用（勾配が小さいほど高得点）
- **選定数**: 水源と同数（デフォルト40箇所）

**3. 発電所候補地の選定**

評価基準：

```
発電所スコア = (標高の低さ) × 1.0 + (勾配の緩やかさ) × 0.5
```

- **標高の低さ**: 低い場所ほど大きな落差が得られる
  - 最大標高 - 現在標高 でスコア化
- **勾配の緩やかさ**: 建設・保守が容易な平坦地を優先
- **選定数**: 水源と同数（デフォルト40箇所）

#### 第3段階：組み合わせの評価と厳選

**全組み合わせの評価**:
- 水源40 × 取水口40 × 発電所40 = 64,000通りの組み合わせを評価
- すべての組み合わせについて発電量を計算（全探索）

**1. 物理的制約のチェック**

各組み合わせが以下の条件を満たすか確認：

```python
# 標高の順序
水源の標高 > 取水口の標高 > 発電所の標高

# 有効落差の確認
有効落差 = 水源標高 - 発電所標高 > 0

# 距離の制約（オプション）
施設間距離が現実的な範囲内
```

**2. 発電量の計算**

物理的制約を満たす組み合わせについて発電量を計算：

```python
# 理論発電量の計算
理論発電量 = 9.8 × 流量(m³/s) × 有効落差(m)

# 効率損失の考慮
- 管路損失: 約10%（摩擦・曲がりなどによる）
- 水車効率: 約85%（水力→回転力の変換）
- 発電機効率: 約95%（回転力→電力の変換）

# 総合効率
総合効率 = (1 - 0.10) × 0.85 × 0.95 ≈ 0.73

# 実際の発電量
実発電量(kW) = 理論発電量 × 0.73
```

簡易計算では総合効率を0.8として概算：

$$発電量(kW) = 9.8 \times 流量(m³/s) \times 有効落差(m) \times 0.8$$

**3. ランキングと上位選出**

- すべての有効な組み合わせを発電量で降順ソート
- 上位N組（デフォルト30組）を最終候補として選出
- 発電量・有効落差・距離などの詳細データを記録

**選出される組み合わせの特徴**:
- 高標高の水源 × 低標高の発電所 = 大きな落差
- 河川に近い水源 = 豊富な流量
- 適度な施設間距離 = 建設コストの現実性

### 最適化のポイント

**なぜ全探索なのか？**:
- 候補数を事前に絞り込んでいるため、組み合わせ数は現実的な範囲
- 全探索により真の最適解を保証
- 計算時間は数秒～数十秒程度（許容範囲内）

**もし計算が遅い場合**:
```python
# 候補数を減らして高速化
selector.run_analysis(candidates_per_type=20)  # 20³ = 8,000通り
```

### 発電量の計算方法

$$発電量(kW) = 9.8 \times 流量(m³/s) \times 有効落差(m) \times 効率$$

- **流量**: 探索範囲内の実際の河川データから推定（通常 0.3～0.5 m³/s程度）
- **有効落差**: 水源の標高 - 発電所の標高（通常 1000～2000m程度）
- **効率**: 水車や発電機の総合効率（約80%）

### 河川データの取得方法

システムは探索範囲内の河川データを自動的に取得します：

1. **Overpass APIから河川情報を取得**
   - OpenStreetMapの河川データ（river, stream）を検索
   - 境界ポリゴンと交差する河川のみを抽出
   - 通常、数千～数万本の河川を検出

2. **流量の推定**
   - 河川タイプ（river: 大河川、stream: 小河川）から基準流量を設定
   - 各候補地点から最も近い河川を探索
   - 距離による減衰を考慮して流量を推定

## 算出手順の詳細（仕様）

### ステップ1：探索範囲の確定

**目的**: 分析対象地域を正確に定義する

**手順**:
1. ユーザーが入力した地名（例：「長野県」）をNominatim APIで検索
2. 該当する行政区画の中心座標（緯度・経度）を取得
3. 行政区画の境界ポリゴンデータを取得
   - 境界は数千～数万個の座標点で構成される多角形
   - 例：長野県の場合、約36,000個の頂点で境界を定義
4. 境界ポリゴンから面積を計算
   - Shapelyライブラリで多角形の面積を算出
   - 緯度1度あたりの距離（約111km）を使って実面積（km²）に変換

**出力データ**:
- 中心座標：(緯度, 経度)
- 境界ポリゴン：[(lat1, lon1), (lat2, lon2), ...]
- バウンディングボックス：(最小緯度, 最大緯度, 最小経度, 最大経度)
- 面積：○○ km²

---

### ステップ2：探索グリッドの生成

**目的**: 地域全体を均等にカバーする評価ポイントを配置

**手順**:
1. **グリッドサイズの決定**
   - 自動モード：面積 ÷ 目標密度(15 km²/点) の平方根
   - 手動指定：ユーザーが指定した値を使用
   - 例：面積が13,500 km²の場合 → √(13500/15) = 30 → 30×30グリッド

2. **グリッドポイントの配置**
   - バウンディングボックス内を等間隔に分割
   - 緯度方向にN個、経度方向にN個の点を配置
   - 総候補点数 = N × N 個

3. **境界内フィルタリング**
   - 各グリッドポイントが境界ポリゴン内に含まれるかチェック
   - Shapely の `polygon.contains(point)` で判定
   - 境界外の点は除外（通常30～60%が除外される）

**出力データ**:
- 境界内グリッドポイント：[(lat1, lon1), (lat2, lon2), ...] 通常100～500点程度

---

### ステップ3：標高データの取得

**目的**: 各グリッドポイントの標高を取得し、地形を把握

**手順**:
1. **API呼び出しの準備**
   - グリッドポイントを150点ずつのバッチに分割
   - Open-Elevation APIは1回のリクエストで複数点の標高を返す

2. **標高データの取得**
   - 各バッチについてAPIリクエストを送信
   - 座標(緯度, 経度)を送り、標高(メートル)を受信
   - リトライ機能：タイムアウト時は再試行

3. **標高データの整理**
   - グリッドポイントと標高を対応付け
   - 標高の統計量を計算（最小値、最大値、平均値、中央値）

**出力データ**:
- 標高配列：[elev1, elev2, elev3, ...] 各グリッドポイントに対応
- 標高統計：最小○○m、最大○○m、平均○○m

---

### ステップ4：河川データの取得

**目的**: 水源候補地の流量を推定するための河川情報を収集

**手順**:
1. **Overpass APIクエリの構築**
   ```
   [way["waterway"~"river|stream"](poly:"境界ポリゴンの座標列")]
   ```
   - 河川タイプを「river」（大河川）と「stream」（小河川）に限定
   - 境界ポリゴンと交差する河川のみを取得

2. **河川データの解析**
   - 各河川の座標リスト（緯度・経度の配列）を取得
   - 河川のタイプ（river/stream）を記録
   - 河川名があれば記録

3. **境界内フィルタリング**
   - 河川の座標が境界ポリゴンと交差するかチェック
   - Shapely の `polygon.intersects(linestring)` で判定
   - 交差しない河川は除外

4. **河川の統計**
   - タイプ別の河川数を集計
   - 平均流量の推定（後述）

**出力データ**:
- 河川リスト：[{type: "river", coords: [...], name: "..."}, ...]
- 河川数：river: ○○本、stream: ○○本、合計○○本

---

### ステップ5：地形勾配の計算

**目的**: 水の流れやすさ、建設の難易度を評価

**手順**:
1. **隣接点の標高差を計算**
   - 各グリッドポイントについて、周囲8方向の隣接点を探索
   - 隣接点との標高差を計算
   - 距離で除算して勾配（m/m）を算出

2. **勾配の合成**
   - 8方向の勾配ベクトルを合成
   - 合成勾配の大きさ：地形の急峻さ
   - 合成勾配の向き：水が流れる方向

3. **勾配の正規化**
   - 全ポイントの勾配を0～1の範囲に正規化
   - 最小勾配 = 0（平坦）、最大勾配 = 1（急峻）

**出力データ**:
- 勾配配列：[grad1, grad2, ...] 各グリッドポイントに対応
- 勾配範囲：最小○○、最大○○

---

### ステップ6：水源候補地の選定

**目的**: 高標高・高流量の地点を水源候補として選定

**手順**:
1. **流量の推定**（各グリッドポイントについて）
   - 最も近い河川を探索（通常5km以内）
   - 河川までの距離 d (km) を計算
   - 河川タイプから基準流量を設定
     - river: Q_base = 1.0 m³/s
     - stream: Q_base = 0.5 m³/s
   - 距離減衰を適用：Q = Q_base × exp(-d / 2.0)
   - 河川が見つからない場合：Q = 0.1 m³/s（最小値）

2. **水源スコアの計算**（各グリッドポイントについて）
   ```
   正規化標高 = (標高 - 最小標高) / (最大標高 - 最小標高)
   正規化流量 = (流量 - 最小流量) / (最大流量 - 最小流量)
   正規化勾配 = (勾配 - 最小勾配) / (最大勾配 - 最小勾配)
   
   水源スコア = 正規化標高 × 1.0 + 正規化流量 × 0.3 + 正規化勾配 × 0.2
   ```
   - 重み付けの意味：
     - 標高（×1.0）：最重要要素。高いほど落差が大きい
     - 流量（×0.3）：水量の確保。多いほど発電量が増える
     - 勾配（×0.2）：水の集まりやすさ。適度な勾配が望ましい

3. **上位候補の選定**
   - すべてのグリッドポイントをスコアで降順ソート
   - 上位N個（デフォルト40個）を水源候補として選定

**出力データ**:
- 水源候補リスト：[{lat, lon, elevation, flow, score}, ...]
- 候補数：○○箇所
- 平均標高：○○m、平均流量：○○ m³/s

---

### ステップ7：取水口候補地の選定

**目的**: 中間標高・平坦地を取水口候補として選定

**手順**:
1. **取水口スコアの計算**（各グリッドポイントについて）
   ```
   中間度 = 1 - |2 × 正規化標高 - 1|
     # 標高が中間(0.5)に近いほど1に近づく
     # 例：標高0.5 → 中間度1.0、標高0.0 or 1.0 → 中間度0.0
   
   平坦度 = 1 / (1 + 正規化勾配)
     # 勾配が小さいほど1に近づく
     # 例：勾配0 → 平坦度1.0、勾配∞ → 平坦度0.0
   
   取水口スコア = 中間度 × 1.0 + 平坦度 × 0.5
   ```
   - 重み付けの意味：
     - 中間度（×1.0）：水源と発電所の中間標高が理想
     - 平坦度（×0.5）：建設しやすさ、保守の容易さ

2. **上位候補の選定**
   - すべてのグリッドポイントをスコアで降順ソート
   - 上位N個（水源と同数）を取水口候補として選定

**出力データ**:
- 取水口候補リスト：[{lat, lon, elevation, score}, ...]
- 候補数：○○箇所
- 平均標高：○○m

---

### ステップ8：発電所候補地の選定

**目的**: 低標高・平坦地を発電所候補として選定

**手順**:
1. **発電所スコアの計算**（各グリッドポイントについて）
   ```
   低標高度 = 1 - 正規化標高
     # 標高が低いほど1に近づく
     # 例：最低標高 → 1.0、最高標高 → 0.0
   
   平坦度 = 1 / (1 + 正規化勾配)
     # 取水口と同じ計算
   
   発電所スコア = 低標高度 × 1.0 + 平坦度 × 0.5
   ```
   - 重み付けの意味：
     - 低標高度（×1.0）：低いほど落差が大きい
     - 平坦度（×0.5）：建設・保守の容易さ

2. **上位候補の選定**
   - すべてのグリッドポイントをスコアで降順ソート
   - 上位N個（水源と同数）を発電所候補として選定

**出力データ**:
- 発電所候補リスト：[{lat, lon, elevation, score}, ...]
- 候補数：○○箇所
- 平均標高：○○m

---

### ステップ9：組み合わせの評価

**目的**: すべての組み合わせについて発電量を計算し、ランク付け

**手順**:
1. **組み合わせの生成**
   - 水源候補 × 取水口候補 × 発電所候補
   - 総組み合わせ数 = N_水源 × N_取水口 × N_発電所
   - 例：40 × 40 × 40 = 64,000通り

2. **物理的制約のチェック**（各組み合わせについて）
   ```
   # 標高の順序チェック
   if 水源標高 <= 取水口標高:
       この組み合わせは無効 → スキップ
   if 取水口標高 <= 発電所標高:
       この組み合わせは無効 → スキップ
   
   # 有効落差の計算
   有効落差 = 水源標高 - 発電所標高
   
   if 有効落差 <= 0:
       この組み合わせは無効 → スキップ
   ```

3. **発電量の計算**（有効な組み合わせについて）
   ```
   # 基本パラメータ
   重力加速度 g = 9.8 m/s²
   流量 Q = 水源の推定流量 (m³/s)
   有効落差 H = 水源標高 - 発電所標高 (m)
   
   # 理論水力 (kW)
   理論水力 = g × Q × H / 1000
   
   # 損失の考慮
   管路損失係数 = 0.90  # 管路摩擦、曲がりなどで10%損失
   水車効率 = 0.85      # ペルトン水車などで85%程度
   発電機効率 = 0.95    # 発電機で95%程度
   
   総合効率 = 管路損失係数 × 水車効率 × 発電機効率
            = 0.90 × 0.85 × 0.95
            = 0.727 ≈ 0.73
   
   # 簡易計算では0.8を使用
   簡易効率 = 0.80
   
   # 実発電量 (kW)
   発電量 = 理論水力 × 簡易効率
          = 9.8 × Q × H × 0.8 / 1000
          = 0.00784 × Q × H  (kW)
   ```

4. **距離の計算**（参考情報として）
   ```
   # 緯度経度から距離を計算（Haversine公式）
   水源-取水口間距離 = haversine(水源座標, 取水口座標)
   取水口-発電所間距離 = haversine(取水口座標, 発電所座標)
   水源-発電所間直線距離 = haversine(水源座標, 発電所座標)
   ```

5. **組み合わせデータの記録**
   ```
   各組み合わせについて以下を記録：
   - 水源の情報（座標、標高、流量）
   - 取水口の情報（座標、標高）
   - 発電所の情報（座標、標高）
   - 有効落差 (m)
   - 発電量 (kW)
   - 各施設間の距離 (km)
   ```

**出力データ**:
- 有効な組み合わせリスト（発電量 > 0のもの）
- 評価済み組み合わせ数：○○通り

---

### ステップ10：ランキングと最適解の選出

**目的**: 発電量の高い組み合わせを選出

**手順**:
1. **ソート**
   - すべての有効な組み合わせを発電量で降順ソート
   - 最大発電量の組み合わせが第1位に

2. **上位候補の選出**
   - 上位M個（デフォルト30個）を最終候補として選出
   - 選出理由：
     - 上位候補は複数の選択肢を提供
     - 現地条件（地権、環境など）で選択可能

3. **結果の整理**
   ```
   各候補について：
   - 順位 (1位、2位、...)
   - 発電量 (kW)
   - 有効落差 (m)
   - 水源・取水口・発電所の詳細座標と標高
   - 施設間距離
   ```

**出力データ**:
- 最終候補リスト：上位M個の組み合わせ
- 第1位の発電量：最大○○ kW
- 平均発電量：○○ kW

---

### ステップ11：可視化とファイル出力

**目的**: 結果を分かりやすく表示・保存

**可視化**:
1. **地図の生成**
   - Foliumライブラリで対話型地図を作成
   - 上位3組の候補地をプロット
     - 赤・青・緑でマーカーを色分け
     - 施設間を線で接続
   - 全グリッドポイントを標高で色分け表示（背景）

2. **グラフの生成**
   - 標高分布図：グリッドポイントの標高を散布図で表示
   - 発電量比較図：上位候補の発電量を棒グラフで比較
   - 有効落差比較図：上位候補の落差を棒グラフで比較
   - 標高プロファイル図：上位3組の施設配置を線グラフで表示

**ファイル出力**:
1. **HTMLファイル**：対話型地図を保存
2. **CSVファイル**：全候補の詳細データを保存
3. **PNGファイル**：各グラフを画像として保存
4. **テキストファイル**：分析結果のサマリーを保存

**出力データ**:
- ファイル保存先：`deta/<日時>/`
- ファイル数：通常7個（HTML×1、CSV×1、PNG×4、TXT×1）

---

## 算出結果の例

### 長野県の場合

**入力**:
- 地名：「長野県」
- グリッドサイズ：28×28（自動設定）
- 候補数：各40箇所（自動設定）

**ステップ別の出力**:
1. **範囲確定**：面積 13,509 km²、境界頂点数 36,137個
2. **グリッド生成**：784点 → 378点（境界内）
3. **標高取得**：325m ～ 2,857m、平均 1,149m
4. **河川取得**：26,193本（river: 5,128本、stream: 21,065本）
5. **勾配計算**：5.0 ～ 1,998.0
6. **水源選定**：40箇所、平均標高 2,003m、平均流量 0.45 m³/s
7. **取水口選定**：40箇所、平均標高 1,517m
8. **発電所選定**：40箇所、平均標高 488m
9. **組み合わせ評価**：64,000通り評価
10. **最適解選出**：上位30組、最大発電量 1,146 kW

**実行時間**：約97秒

---

## 出力されるファイル

結果は `deta/日時/` フォルダに保存されます：

| ファイル名 | 内容 |
|----------|------|
| `hydro_map_top3_<地名>_<日時>.html` | 上位3つの候補地を表示した地図（英語表記） |
| `hydro_sites_<地名>_<日時>.csv` | すべての候補地の詳細データ（英語列名） |
| `1_Elevation_Distribution_<地名>_<日時>.png` | 標高分布グラフ |
| `2_Power_Output_Comparison_<地名>_<日時>.png` | 発電量比較グラフ |
| `3_Effective_Head_Comparison_<地名>_<日時>.png` | 有効落差比較グラフ |
| `4_Facility_Elevation_Profile_<地名>_<日時>.png` | 施設配置プロファイル |
| `summary_<地名>_<日時>.txt` | 分析結果のサマリー（日本語） |

※ ファイル名の地名部分は日本語、グラフのタイトルやラベルは英語で保存されます

## 詳細設定（任意）

もっと細かく調整したい場合：

```python
selector.run_analysis(
    grid_size=30,           # 探索グリッドの密度（大きいほど詳細、計算時間増）
    candidates_per_type=40, # 各タイプの候補数（多いほど選択肢が増える）
    top_n=30               # 最終的に表示する候補数
)
```

※ 通常は自動で最適な値が設定されるので、指定しなくてOKです

**自動設定のロジック**:
- `grid_size`: 面積に応じて自動計算（目標密度: 約15 km²/点）
- `candidates_per_type`: グリッド点数の5%（最小20、最大80）
- `top_n`: 組合せ総数に応じて10～30を自動選択

## 結果の見方

### 地図の見方
- **マーカーの色**で候補地の順位が分かります
  - 赤：第1位、青：第2位、緑：第3位
- **線**で施設同士がつながっています（水の流れる経路）
- マーカーをクリックすると詳細情報が表示されます
  - 水源：標高、推定流量
  - 取水口：標高
  - 発電所：標高、発電量

### CSVデータの列
- `Rank`: 順位
- `Power_kW`: 推定発電量（キロワット）
- `Effective_Head_m`: 有効落差（メートル）
- `WS_Lat`, `WS_Lon`: 水源の緯度・経度
- `WS_Elevation_m`: 水源の標高
- `WS_Flow_m3_s`: 水源の推定流量（m³/s）
- `Intake_Lat`, `Intake_Lon`: 取水口の緯度・経度
- `Intake_Elevation_m`: 取水口の標高
- `PH_Lat`, `PH_Lon`: 発電所の緯度・経度
- `PH_Elevation_m`: 発電所の標高
- `WS_Intake_Distance_km`: 水源-取水口間の距離
- `Intake_PH_Distance_km`: 取水口-発電所間の距離
- `WS_PH_Distance_km`: 水源-発電所間の直線距離

## 注意事項

- このシステムの結果は**参考値**です
- 実際の建設には、詳細な現地調査や許可申請が必要です
- 計算時間は地域の広さによって変わります
  - 小規模な市町村：数分程度
  - 県レベル：数十分～1時間程度
- インターネット接続が必要です（地図や標高データを取得するため）
- 河川流量は推定値であり、実際の測定値とは異なる場合があります

## トラブルシューティング

### エラーが出た場合
1. **地名が見つからない**: 正式な市区町村名・都道府県名を入力してください
2. **タイムアウトエラー**: インターネット接続を確認してください
3. **計算が終わらない**: grid_sizeを小さくしてみてください（例: 20）

### 実行が遅い場合
```python
# 高速モード（精度は少し下がります）
selector.run_analysis(grid_size=15, candidates_per_type=20, top_n=10)
```

## 技術的な詳細

### 使用API
- **Nominatim API**: 地名から座標と行政区画境界を取得
- **Open-Elevation API**: 標高データの一括取得
- **Overpass API**: OpenStreetMapから河川データを取得

### データ精度
- **境界精度**: 行政区画の面積誤差は通常1%未満
- **標高精度**: ±数メートル程度（地形データの解像度に依存）
- **流量推定**: 河川タイプと距離から推定（実測値ではない）

### 計算の最適化
- グリッドポイントと河川データを境界内にフィルタリング
- 標高データはバッチ処理で高速取得（150点/リクエスト）
- 組合せ評価は全探索（数万～数十万通り）で最適解を保証


<a href="https://colab.research.google.com/github/tsuka22120/4IE2/blob/main/ED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 水力発電所候補地選定システム

このノートブックは、指定した地域の地形データを分析し、水力発電所の最適な候補地を選定します。

## 機能
- 地名による探索範囲の指定
- 標高データの取得と分析
- 水源・取水口・発電所の候補地選定
- 推定発電量の計算
- 地図とグラフによる可視化
- リアルタイムの進捗表示

In [ ]:
# ============================================================
# セル (DEMO): 小規模デモ実行（grid_size=20, candidates_per_type=20, top_n=5）
# - 解析は別スレッドで実行して、ステータスをポーリング表示します
# - 実行完了後、上位候補を簡単に表示します
# ============================================================
import threading
import time

location_name = "松本市"
print(f"Starting demo for: {location_name}")

# 既にselectorが定義されていれば再利用、なければ新規作成
if 'selector' in globals() and isinstance(globals().get('selector'), HydroSiteSelector):
    print("Reusing existing selector instance")
else:
    selector = HydroSiteSelector(location_name)

result = {}

def run_and_store():
    try:
        m, f = selector.run_analysis(grid_size=20, candidates_per_type=20, top_n=5)
        result['map'] = m
        result['fig'] = f
    except Exception as e:
        result['error'] = str(e)

# スレッドで実行
thread = threading.Thread(target=run_and_store, daemon=True)
thread.start()

# ポーリングでステータス表示
for _ in range(300):
    st = selector.get_status()
    print(f"STATUS: stage={st.get('stage')}, progress={st.get('progress')}%, msg={st.get('message')}")
    if st.get('stage') in ('done', 'error'):
        break
    time.sleep(2)

thread.join(timeout=10)

map_result = result.get('map')
fig_result = result.get('fig')

if 'error' in result:
    print(f"Error during run: {result['error']}")

# 上位候補の概要を表示
if hasattr(selector, 'best_combinations') and selector.best_combinations:
    print("\nTop combinations:")
    for i, combo in enumerate(selector.best_combinations, start=1):
        ws = combo['water_source']
        it = combo['intake']
        ph = combo['powerhouse']
        print(f"#{i}: Power={combo['power_kw']:.1f} kW, Head={combo['head']:.1f} m, WaterFlow(at intake)={it.get('river_flow', 0.0):.3f} m^3/s")
else:
    print("No combinations found or run did not complete.")


In [ ]:
# ============================================================
# セル4: ライブラリのインポート
# ============================================================
# インストールしたライブラリをインポートして使用可能にします。
# 警告メッセージを非表示にして、出力を見やすくします。
# ============================================================

import folium
from folium import plugins
import numpy as np
import pandas as pd
import requests
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import time
from datetime import datetime
from itertools import combinations
import warnings
from shapely.geometry import Point, Polygon, MultiPolygon
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# ログ機能付きprintラッパー
# ============================================================
# すべてのprint出力をファイルにも保存するための機能を追加
# ============================================================

import sys
import os
from datetime import datetime
from io import StringIO

class TeeOutput:
    """標準出力とファイルの両方に出力するクラス"""
    def __init__(self, file_path):
        self.terminal = sys.stdout
        self.log_file = open(file_path, 'w', encoding='utf-8')
        
    def write(self, message):
        self.terminal.write(message)
        self.log_file.write(message)
        self.log_file.flush()
        
    def flush(self):
        self.terminal.flush()
        self.log_file.flush()
        
    def close(self):
        if self.log_file and not self.log_file.closed:
            self.log_file.close()

# グローバル変数として保持
_tee_output = None

def start_logging(location_name="default"):
    """ログ記録を開始"""
    global _tee_output
    
    # 既存のログがあれば閉じる
    if _tee_output:
        stop_logging()
    
    # ログディレクトリの作成
    timestamp = datetime.now().strftime("%Y%m%d%H%M")
    log_dir = f"deta/{timestamp}"
    os.makedirs(log_dir, exist_ok=True)
    
    # ログファイルのパス
    log_filename = f"execution_log_{location_name}_{timestamp}.txt"
    log_path = os.path.join(log_dir, log_filename)
    
    # Tee出力を開始
    _tee_output = TeeOutput(log_path)
    sys.stdout = _tee_output
    
    print(f"{'='*70}")
    print(f"水力発電候補地選定システム - 実行ログ")
    print(f"{'='*70}")
    print(f"地域名: {location_name}")
    print(f"開始時刻: {datetime.now().strftime('%Y年%m月%d日 %H:%M:%S')}")
    print(f"ログファイル: {log_path}")
    print(f"{'='*70}\n")
    
    return log_dir, log_path

def stop_logging():
    """ログ記録を停止"""
    global _tee_output
    
    if _tee_output:
        print(f"\n{'='*70}")
        print(f"実行完了時刻: {datetime.now().strftime('%Y年%m月%d日 %H:%M:%S')}")
        print(f"{'='*70}")
        
        # 標準出力を元に戻す
        sys.stdout = _tee_output.terminal
        _tee_output.close()
        _tee_output = None

print("✓ ログ機能を読み込みました")

## ログ出力機能について

このノートブックでは、プログラムのすべての出力を自動的に`.txt`ファイルに保存します。

### 機能の使い方

```python
# ログ記録を開始
log_dir, log_path = start_logging("地域名")

try:
    # ここに実行したいコード
    # print文はすべて画面とファイルの両方に出力されます
    
finally:
    # ログ記録を停止（必ず実行）
    stop_logging()
```

### 保存されるログファイル

- ファイル名: `execution_log_<地域名>_<日時>.txt`
- 保存先: `deta/<日時>/`フォルダ
- 内容: すべてのprint出力、エラーメッセージ、進捗情報

### 注意事項

- `start_logging()`を呼び出すと、以降のすべての出力が記録されます
- `stop_logging()`を必ず呼び出してファイルを正しく閉じてください
- `try-finally`ブロックを使用することで、エラーが発生してもログファイルが確実に閉じられます

In [ ]:
# ============================================================
# セル5: HydroSiteSelector クラスの定義（パート1 - 最適化版）
# ============================================================
# 主要メソッド: __init__, 座標取得, グリッド生成, 標高取得
# 高速化: 河川近接度計算を削除し、河川流量データベースを使用
# ============================================================

class HydroSiteSelector:
    def __init__(self, location_name):
        """初期化"""
        self.location_name = location_name
        self.center_lat = None
        self.center_lon = None
        self.bbox = None
        self.boundary_polygon = None
        self.grid_points = []
        self.elevation_data = None
        self.candidates = {
            'water_sources': [],
            'intakes': [],
            'powerhouses': []
        }
        self.best_combinations = []
        self.area_km2 = 0
        
        # 河川データ（探索範囲から取得）
        self.river_data = []
        self.river_flow_estimates = {}
        
        # ステータス管理
        self.status = {'stage': 'init', 'progress': 0, 'message': 'Initialized'}
    
    def update_status(self, stage, progress, message):
        """ステータス更新"""
        self.status = {'stage': stage, 'progress': progress, 'message': message}
    
    def get_status(self):
        """現在のステータスを取得"""
        return self.status
    
    def get_location_coordinates(self):
        """地名から座標と境界を取得"""
        print(f"\n{'='*60}")
        print(f"地域: {self.location_name}")
        print(f"{'='*60}\n")
        self.update_status(stage='geocoding', progress=1, message='座標取得中')
        
        print(f"'{self.location_name}' の座標と境界を取得中...")
        
        try:
            from geopy.geocoders import Nominatim
            geolocator = Nominatim(user_agent="hydro_site_selector")
            
            location = geolocator.geocode(self.location_name, language='ja', timeout=10)
            
            if location:
                self.center_lat = location.latitude
                self.center_lon = location.longitude
                print(f"✓ 中心座標: ({self.center_lat:.6f}, {self.center_lon:.6f})")
                
                # 境界ポリゴン取得
                if self._fetch_boundary_polygon():
                    self.update_status(stage='geocoding', progress=5, message='座標取得完了')
                    return True
            
            print(f"エラー: '{self.location_name}' の座標が見つかりません")
            return False
            
        except Exception as e:
            print(f"エラー: 座標取得に失敗 - {e}")
            return False
    
    def _fetch_boundary_polygon(self):
        """境界ポリゴンを取得（正確な行政区画境界を使用）"""
        print(f"\n'{self.location_name}' の正確な境界を取得中...")
        
        try:
            import requests
            from shapely.geometry import shape, Point, MultiPolygon, Polygon
            import time
            
            # Nominatim APIで境界データ取得（複数の検索方法を試行）
            search_queries = [
                self.location_name,
                f"{self.location_name}, Japan",
                f"{self.location_name}, 日本"
            ]
            
            for query in search_queries:
                url = "https://nominatim.openstreetmap.org/search"
                params = {
                    'q': query,
                    'format': 'json',
                    'polygon_geojson': 1,
                    'limit': 5,  # 複数候補を取得
                    'addressdetails': 1
                }
                
                print(f"  検索中: '{query}'...")
                response = requests.get(url, params=params, timeout=30, 
                                       headers={'User-Agent': 'HydroSiteSelector/1.0'})
                time.sleep(1)  # レート制限対策
                
                if response.status_code == 200:
                    data = response.json()
                    
                    # 最適な結果を選択（行政区画を優先）
                    for result in data:
                        if 'geojson' in result:
                            # 行政区画タイプを優先
                            osm_type = result.get('osm_type', '')
                            class_type = result.get('class', '')
                            
                            # 県、市、町、村などの行政区画を優先
                            if class_type in ['boundary', 'administrative', 'place']:
                                geojson = result['geojson']
                                
                                try:
                                    self.boundary_polygon = shape(geojson)
                                    
                                    # MultiPolygonの場合も対応
                                    if not isinstance(self.boundary_polygon, (Polygon, MultiPolygon)):
                                        continue
                                    
                                    # バウンディングボックス設定
                                    bounds = self.boundary_polygon.bounds
                                    self.bbox = (bounds[1], bounds[3], bounds[0], bounds[2])
                                    
                                    # ポリゴンの実際の面積を計算
                                    # Shapely の area は度数単位なので、km²に変換
                                    center_lat = (bounds[1] + bounds[3]) / 2
                                    lat_km_per_deg2 = 111.0 * 111.0 * np.cos(np.radians(center_lat))
                                    self.area_km2 = abs(self.boundary_polygon.area) * lat_km_per_deg2
                                    
                                    # 境界の頂点数を取得
                                    if isinstance(self.boundary_polygon, Polygon):
                                        vertex_count = len(self.boundary_polygon.exterior.coords)
                                    elif isinstance(self.boundary_polygon, MultiPolygon):
                                        vertex_count = sum(len(p.exterior.coords) for p in self.boundary_polygon.geoms)
                                    else:
                                        vertex_count = 0
                                    
                                    print(f"\n✓ 正確な行政区画境界を取得しました")
                                    print(f"  地域名: {result.get('display_name', self.location_name)}")
                                    print(f"  タイプ: {result.get('type', 'unknown')}")
                                    print(f"  緯度範囲: {bounds[1]:.6f}° ~ {bounds[3]:.6f}°")
                                    print(f"  経度範囲: {bounds[0]:.6f}° ~ {bounds[2]:.6f}°")
                                    print(f"  境界頂点数: {vertex_count:,}")
                                    print(f"  実面積: {self.area_km2:.1f} km²")
                                    
                                    return True
                                    
                                except Exception as e:
                                    print(f"  警告: ポリゴン処理エラー - {e}")
                                    continue
            
            # すべての検索で境界が見つからなかった場合
            print(f"\n✗ エラー: '{self.location_name}' の正確な境界データが見つかりません")
            print("  対策:")
            print("    1. 地域名のスペルを確認してください")
            print("    2. より具体的な地名を使用してください（例: '長野県', '松本市'）")
            print("    3. OpenStreetMapにその地域の境界データが登録されているか確認してください")
            return False
            
        except Exception as e:
            print(f"\n✗ エラー: 境界取得に失敗 - {e}")
            import traceback
            traceback.print_exc()
            return False
    
    def is_point_in_boundary(self, lat, lon):
        """ポイントが正確な境界内かチェック（厳密版）"""
        if self.boundary_polygon:
            try:
                from shapely.geometry import Point
                return self.boundary_polygon.contains(Point(lon, lat))
            except Exception as e:
                print(f"警告: 境界チェックエラー ({lat}, {lon}): {e}")
                return False
        
        # 境界ポリゴンがない場合はエラー
        print(f"エラー: 境界ポリゴンが設定されていません")
        return False
    
    def generate_grid_points(self, grid_size=30):
        """グリッドポイント生成（正確な境界を使用）"""
        print(f"\nグリッドポイント生成中 ({grid_size}x{grid_size})...")
        self.update_status(stage='generate_grid', progress=8, message=f'{grid_size}x{grid_size}グリッド生成中')
        
        if not self.boundary_polygon:
            print("エラー: 境界ポリゴンが設定されていません")
            return []
        
        min_lat, max_lat, min_lon, max_lon = self.bbox
        
        lats = np.linspace(min_lat, max_lat, grid_size)
        lons = np.linspace(min_lon, max_lon, grid_size)
        
        # メッシュグリッド作成
        lat_mesh, lon_mesh = np.meshgrid(lats, lons)
        all_points = list(zip(lat_mesh.flatten(), lon_mesh.flatten()))
        
        print(f"  バウンディングボックス内の候補点: {len(all_points)}")
        print(f"  正確な境界内にフィルタリング中...")
        
        # 境界ポリゴン内のポイントのみを選択
        from shapely.geometry import Point
        self.grid_points = []
        
        for i, p in enumerate(all_points):
            if i % 100 == 0 and i > 0:
                print(f"    進捗: {i}/{len(all_points)} ({i/len(all_points)*100:.1f}%)", end='\r')
            
            if self.is_point_in_boundary(p[0], p[1]):
                self.grid_points.append(p)
        
        print(f"\n✓ {len(self.grid_points)}個のグリッドポイントを生成（境界内のみ）")
        print(f"  除外されたポイント: {len(all_points) - len(self.grid_points)}")
        self.update_status(stage='generate_grid', progress=10, message=f'{len(self.grid_points)}ポイント生成完了')
        return self.grid_points
    
    def get_elevation(self, lat, lon, retries=2):
        """単一ポイントの標高取得"""
        import time
        for attempt in range(retries):
            try:
                url = f"https://api.open-elevation.com/api/v1/lookup?locations={lat},{lon}"
                response = requests.get(url, timeout=5)
                if response.status_code == 200:
                    data = response.json()
                    return data['results'][0]['elevation']
                time.sleep(0.5)
            except:
                if attempt == retries - 1:
                    return None
                time.sleep(1)
        return None
    
    def fetch_elevation_data(self, batch_size=100):
        """標高データ取得（最適化版・バッチサイズ増）"""
        import time
        print(f"\n標高データ取得中...")
        self.update_status(stage='fetch_elevation', progress=12, message='標高データ取得開始')
        
        self.elevation_data = np.zeros(len(self.grid_points))
        
        from tqdm.notebook import tqdm
        with tqdm(total=len(self.grid_points), desc="標高データ取得") as pbar:
            for i in range(0, len(self.grid_points), batch_size):
                batch = self.grid_points[i:i+batch_size]
                
                try:
                    # バッチリクエスト
                    locations = '|'.join([f"{lat},{lon}" for lat, lon in batch])
                    url = f"https://api.open-elevation.com/api/v1/lookup?locations={locations}"
                    
                    response = requests.get(url, timeout=30)
                    
                    if response.status_code == 200:
                        results = response.json()['results']
                        for j, result in enumerate(results):
                            self.elevation_data[i+j] = result['elevation']
                    else:
                        # エラー時は個別取得
                        for j, (lat, lon) in enumerate(batch):
                            elev = self.get_elevation(lat, lon)
                            if elev is not None:
                                self.elevation_data[i+j] = elev
                    
                    time.sleep(0.3)  # API制限対策（短縮）
                    
                except Exception as e:
                    # エラー時は個別取得
                    for j, (lat, lon) in enumerate(batch):
                        elev = self.get_elevation(lat, lon)
                        if elev is not None:
                            self.elevation_data[i+j] = elev
                
                pbar.update(len(batch))
                
                # 進捗更新
                processed = min(i + len(batch), len(self.grid_points))
                pct = int(12 + (processed / len(self.grid_points)) * 18)  # 12%~30%
                self.update_status(stage='fetch_elevation', progress=pct, 
                                 message=f'標高取得 {processed}/{len(self.grid_points)}')
        
        print(f"✓ 標高データ取得完了")
        print(f"  標高範囲: {self.elevation_data.min():.1f}m ~ {self.elevation_data.max():.1f}m")
        self.update_status(stage='fetch_elevation', progress=30, message='標高取得完了')
        return self.elevation_data
    
    def fetch_river_data(self):
        """探索範囲内の河川データを取得（正確な境界を使用）"""
        import time
        print(f"\n河川データ取得中（正確な境界内のみ）...")
        self.update_status(stage='fetch_rivers', progress=32, message='河川データ取得中')
        
        try:
            from shapely.geometry import LineString, Point
            
            min_lat, max_lat, min_lon, max_lon = self.bbox
            
            # Overpass API クエリ（バウンディングボックスで広めに取得）
            overpass_url = "http://overpass-api.de/api/interpreter"
            overpass_query = f"""
            [out:json][timeout:60];
            (
              way["waterway"="river"]({min_lat},{min_lon},{max_lat},{max_lon});
              way["waterway"="stream"]({min_lat},{min_lon},{max_lat},{max_lon});
            );
            out geom;
            """
            
            print(f"  Overpass APIに問い合わせ中...")
            response = requests.post(overpass_url, data={'data': overpass_query}, timeout=60)
            
            if response.status_code == 200:
                data = response.json()
                elements = data.get('elements', [])
                
                print(f"  取得した河川候補: {len(elements)}")
                print(f"  正確な境界内にフィルタリング中...")
                
                filtered_count = 0
                for element in elements:
                    if 'geometry' in element:
                        coords = [(node['lon'], node['lat']) for node in element['geometry']]
                        if len(coords) >= 2:
                            line = LineString(coords)
                            
                            # 河川が境界と交差するか、境界内に完全に含まれるかチェック
                            if self.boundary_polygon:
                                if self.boundary_polygon.intersects(line):
                                    # 境界内の部分のみを抽出
                                    try:
                                        intersection = self.boundary_polygon.intersection(line)
                                        if not intersection.is_empty:
                                            river_info = {
                                                'id': element.get('id'),
                                                'name': element.get('tags', {}).get('name', 'unnamed'),
                                                'type': element.get('tags', {}).get('waterway', 'river'),
                                                'geometry': line,  # 元のジオメトリを保持
                                                'width': self._estimate_river_width(element.get('tags', {}))
                                            }
                                            self.river_data.append(river_info)
                                            filtered_count += 1
                                    except Exception as e:
                                        # ジオメトリエラーの場合はスキップ
                                        continue
                            else:
                                # 境界ポリゴンがない場合（エラー状態）
                                river_info = {
                                    'id': element.get('id'),
                                    'name': element.get('tags', {}).get('name', 'unnamed'),
                                    'type': element.get('tags', {}).get('waterway', 'river'),
                                    'geometry': line,
                                    'width': self._estimate_river_width(element.get('tags', {}))
                                }
                                self.river_data.append(river_info)
                
                print(f"✓ {len(self.river_data)}本の河川データを取得")
                
                # 河川流量を推定
                
                print(f"  境界内の河川数: {len(self.river_data)} ({filtered_count}本が境界と交差)")
                print(f"  除外された河川: {len(elements) - filtered_count}")
                
                # 河川タイプの統計
                river_types = {}
                for river in self.river_data:
                    rtype = river['type']
                    river_types[rtype] = river_types.get(rtype, 0) + 1
                
                print(f"  河川タイプ内訳:")
                for rtype, count in river_types.items():
                    print(f"    {rtype}: {count}本")
                
                # 河川流量推定
                self._estimate_river_flows()
                
                self.update_status(stage='fetch_rivers', progress=35, message='河川データ取得完了')
                return True
            else:
                print(f"警告: 河川データの取得に失敗 (ステータス: {response.status_code})")
                return False
        except Exception as e:
            print(f"警告: 河川データ取得エラー - {e}")
            return False
    
    def _estimate_river_width(self, tags):
        """河川タグから幅を推定"""
        # タグから幅情報を取得
        if 'width' in tags:
            try:
                return float(tags['width'])
            except:
                pass
        
        # タイプから推定
        waterway_type = tags.get('waterway', 'stream')
        width_estimates = {
            'river': 20.0,
            'stream': 5.0,
            'canal': 10.0
        }
        return width_estimates.get(waterway_type, 5.0)
    
    def _estimate_river_flows(self):
        """河川の流量を推定"""
        print(f"  河川流量を推定中...")
        
        for river in self.river_data:
            # 幅と長さから簡易的に流量を推定
            length_km = river['geometry'].length * 111  # 度→km変換（概算）
            width_m = river['width']
            
            # 流域面積の推定（非常に簡易的）
            drainage_area_km2 = length_km * width_m * 0.01  # 簡易推定
            
            # 流量推定式（経験式を簡略化）
            # Q(m³/s) ≈ 0.5 * drainage_area(km²)^0.7
            estimated_flow = 0.5 * (drainage_area_km2 ** 0.7)
            estimated_flow = max(0.5, min(estimated_flow, 50.0))  # 0.5～50m³/s の範囲
            
            river_id = river['id']
            self.river_flow_estimates[river_id] = estimated_flow
        
        if self.river_data:
            avg_flow = np.mean(list(self.river_flow_estimates.values()))
            print(f"  推定平均流量: {avg_flow:.2f} m³/s")

print("✓ HydroSiteSelector クラス（パート1）定義完了")


In [ ]:
# ============================================================
# セル6: HydroSiteSelector クラスの定義（パート2）
# ============================================================
# 候補地選定メソッド（実際の河川データを使用）
# ============================================================

def get_river_flow_for_point(self, lat, lon):
    """指定地点に最も近い河川の流量を取得"""
    from shapely.geometry import Point
    
    if not self.river_data:
        # 河川データがない場合はデフォルト値
        return 3.0
    
    point = Point(lon, lat)
    
    # 最も近い河川を検索
    min_distance = float('inf')
    closest_river_id = None
    
    for river in self.river_data:
        distance = point.distance(river['geometry'])
        if distance < min_distance:
            min_distance = distance
            closest_river_id = river['id']
    
    # 距離による減衰を適用（15km以内を有効範囲とする）
    distance_km = min_distance * 111  # 度→km変換
    if distance_km > 15:
        # 遠すぎる場合はデフォルト値
        return 1.0
    
    # 距離減衰係数
    decay_factor = np.exp(-distance_km / 5)
    
    # 流量取得
    base_flow = self.river_flow_estimates.get(closest_river_id, 3.0)
    
    return base_flow * decay_factor

def get_average_river_flow(self):
    """探索範囲全体の平均河川流量を取得"""
    if not self.river_flow_estimates:
        return 3.0
    
    avg_flow = np.mean(list(self.river_flow_estimates.values()))
    print(f"  探索範囲の平均河川流量: {avg_flow:.2f} m³/s")
    print(f"  （検出河川数: {len(self.river_data)}本）")
    
    return avg_flow

def calculate_slope_optimized(self):
    """勾配計算（最適化版・シンプル）"""
    print(f"\n勾配計算中...")
    
    # グリッドサイズ推定
    n = len(self.grid_points)
    grid_size = int(np.sqrt(n))
    
    # 2D配列への変換（パディングなし、近似）
    if grid_size ** 2 != n:
        # 最も近い完全平方数
        grid_size = int(np.ceil(np.sqrt(n)))
        padded = np.pad(self.elevation_data, (0, grid_size**2 - n), mode='edge')
    else:
        padded = self.elevation_data
    
    elev_2d = padded.reshape(grid_size, grid_size)
    
    # 勾配計算（NumPyの高速gradient）
    grad_y, grad_x = np.gradient(elev_2d)
    slope_2d = np.sqrt(grad_x**2 + grad_y**2)
    
    # 1D配列に戻す
    slope = slope_2d.flatten()[:n]
    
    print(f"✓ 勾配計算完了 (範囲: {slope.min():.3f} ~ {slope.max():.3f})")
    return slope

def find_water_sources(self, top_n=50):
    """水源候補選定（実際の河川データを使用）"""
    print(f"\n水源候補地選定中 (上位{top_n}箇所)...")
    self.update_status(stage='water_source', progress=40, message='水源候補選定中')
    
    # 平均河川流量取得
    base_flow = self.get_average_river_flow()
    
    # 勾配計算
    slope = self.calculate_slope_optimized()
    
    # 正規化
    elev_norm = (self.elevation_data - self.elevation_data.min()) / \
                max(1e-6, (self.elevation_data.max() - self.elevation_data.min()))
    slope_norm = (slope - slope.min()) / max(1e-6, (slope.max() - slope.min()))
    
    # スコアリング: 高標高(60%) + 適度な勾配(40%)
    scores = 0.6 * elev_norm + 0.4 * slope_norm
    
    # 境界内フィルタリング
    valid_indices = [idx for idx, (lat, lon) in enumerate(self.grid_points) 
                    if self.is_point_in_boundary(lat, lon)]
    
    print(f"  境界内ポイント: {len(valid_indices)}/{len(self.grid_points)}")
    
    # 上位選定
    valid_scores = [(idx, scores[idx]) for idx in valid_indices]
    valid_scores.sort(key=lambda x: x[1], reverse=True)
    top_indices = [idx for idx, _ in valid_scores[:top_n]]
    
    # 候補保存（各地点の河川流量を取得）
    for idx in top_indices:
        lat, lon = self.grid_points[idx]
        point_flow = self.get_river_flow_for_point(lat, lon)
        
        self.candidates['water_sources'].append({
            'lat': lat,
            'lon': lon,
            'elevation': float(self.elevation_data[idx]),
            'score': float(scores[idx]),
            'estimated_flow': point_flow,
            'type': 'water_source'
        })
    
    avg_elev = np.mean([c['elevation'] for c in self.candidates['water_sources']])
    avg_flow = np.mean([c['estimated_flow'] for c in self.candidates['water_sources']])
    print(f"✓ {len(self.candidates['water_sources'])}箇所選定完了")
    print(f"  平均標高: {avg_elev:.1f}m, 平均推定流量: {avg_flow:.2f}m³/s")
    
    self.update_status(stage='water_source', progress=50, message=f'{top_n}箇所選定完了')
    return self.candidates['water_sources']

def find_intakes(self, top_n=50):
    """取水口候補選定（中間標高・緩勾配）"""
    print(f"\n取水口候補地選定中 (上位{top_n}箇所)...")
    self.update_status(stage='intake', progress=50, message='取水口候補選定中')
    
    slope = self.calculate_slope_optimized()
    
    # 正規化
    elev_norm = (self.elevation_data - self.elevation_data.min()) / \
                max(1e-6, (self.elevation_data.max() - self.elevation_data.min()))
    slope_norm = (slope - slope.min()) / max(1e-6, (slope.max() - slope.min()))
    
    # 中間標高スコア（25%～75%が最適）
    middle_score = 1 - 4 * np.abs(elev_norm - 0.5)**2
    middle_score = np.maximum(middle_score, 0)
    
    # 緩勾配スコア
    gentle_slope = 1 - slope_norm
    
    # スコアリング: 中間標高(60%) + 緩勾配(40%)
    scores = 0.6 * middle_score + 0.4 * gentle_slope
    
    # 境界内フィルタリング
    valid_indices = [idx for idx, (lat, lon) in enumerate(self.grid_points) 
                    if self.is_point_in_boundary(lat, lon)]
    
    # 上位選定
    valid_scores = [(idx, scores[idx]) for idx in valid_indices]
    valid_scores.sort(key=lambda x: x[1], reverse=True)
    top_indices = [idx for idx, _ in valid_scores[:top_n]]
    
    # 候補保存
    for idx in top_indices:
        lat, lon = self.grid_points[idx]
        self.candidates['intakes'].append({
            'lat': lat,
            'lon': lon,
            'elevation': float(self.elevation_data[idx]),
            'score': float(scores[idx]),
            'type': 'intake'
        })
    
    avg_elev = np.mean([c['elevation'] for c in self.candidates['intakes']])
    print(f"✓ {len(self.candidates['intakes'])}箇所選定完了")
    print(f"  平均標高: {avg_elev:.1f}m")
    
    self.update_status(stage='intake', progress=55, message=f'{top_n}箇所選定完了')
    return self.candidates['intakes']

def find_powerhouses(self, top_n=50):
    """発電所候補選定（低標高・緩勾配）"""
    print(f"\n発電所候補地選定中 (上位{top_n}箇所)...")
    self.update_status(stage='powerhouse', progress=60, message='発電所候補選定中')
    
    slope = self.calculate_slope_optimized()
    
    # 正規化
    elev_norm = (self.elevation_data - self.elevation_data.min()) / \
                max(1e-6, (self.elevation_data.max() - self.elevation_data.min()))
    slope_norm = (slope - slope.min()) / max(1e-6, (slope.max() - slope.min()))
    
    # スコアリング: 低標高(70%) + 緩勾配(30%)
    scores = 0.7 * (1 - elev_norm) + 0.3 * (1 - slope_norm)
    
    # 境界内フィルタリング
    valid_indices = [idx for idx, (lat, lon) in enumerate(self.grid_points) 
                    if self.is_point_in_boundary(lat, lon)]
    
    # 上位選定
    valid_scores = [(idx, scores[idx]) for idx in valid_indices]
    valid_scores.sort(key=lambda x: x[1], reverse=True)
    top_indices = [idx for idx, _ in valid_scores[:top_n]]
    
    # 候補保存
    for idx in top_indices:
        lat, lon = self.grid_points[idx]
        self.candidates['powerhouses'].append({
            'lat': lat,
            'lon': lon,
            'elevation': float(self.elevation_data[idx]),
            'score': float(scores[idx]),
            'type': 'powerhouse'
        })
    
    avg_elev = np.mean([c['elevation'] for c in self.candidates['powerhouses']])
    print(f"✓ {len(self.candidates['powerhouses'])}箇所選定完了")
    print(f"  平均標高: {avg_elev:.1f}m")
    
    self.update_status(stage='powerhouse', progress=65, message=f'{top_n}箇所選定完了')
    return self.candidates['powerhouses']

# メソッドをクラスに追加
HydroSiteSelector.get_river_flow_for_point = get_river_flow_for_point
HydroSiteSelector.get_average_river_flow = get_average_river_flow
HydroSiteSelector.calculate_slope_optimized = calculate_slope_optimized
HydroSiteSelector.find_water_sources = find_water_sources
HydroSiteSelector.find_intakes = find_intakes
HydroSiteSelector.find_powerhouses = find_powerhouses

print("✓ HydroSiteSelector クラス（パート2）定義完了")


In [ ]:
# ============================================================
# セル7: 発電量推定と最適化（最適化版・河川流量使用）
# ============================================================
# estimate_power_generation: 実河川流量データを使用
# find_best_combinations: 高速組合せ評価
# ============================================================

def estimate_power_generation(self, water_source, intake, powerhouse):
    """発電量推定（河川流量データベース使用）
    
    P = ρ * g * Q * H * η
    - ρ: 水の密度 (1000 kg/m³) → 係数に組み込み
    - g: 重力加速度 (9.8 m/s²)
    - Q: 流量 (m³/s) - 河川流量データベースから
    - H: 有効落差 (m)
    - η: 総合効率 (0.8)
    """
    
    # 物理制約チェック
    if water_source['elevation'] < intake['elevation']:
        return 0
    if intake['elevation'] < powerhouse['elevation']:
        return 0
    
    # 有効落差
    head = water_source['elevation'] - powerhouse['elevation']
    if head <= 0:
        return 0
    
    # 流量: 水源の推定流量を使用
    Q = water_source.get('estimated_flow', 3.0)
    
    # 距離ペナルティ
    from geopy.distance import geodesic
    dist_ws_intake = geodesic(
        (water_source['lat'], water_source['lon']),
        (intake['lat'], intake['lon'])
    ).kilometers
    
    dist_intake_ph = geodesic(
        (intake['lat'], intake['lon']),
        (powerhouse['lat'], powerhouse['lon'])
    ).kilometers
    
    # 流量は距離で減衰（水路損失）
    total_dist = dist_ws_intake + dist_intake_ph
    flow_loss_factor = np.exp(-total_dist / 15)  # 15kmで大幅減衰
    Q_effective = Q * flow_loss_factor
    Q_effective = max(0.1, min(Q_effective, 20.0))  # 0.1~20 m³/s
    
    # 発電量計算 (kW)
    efficiency = 0.8
    power = 9.8 * Q_effective * head * efficiency
    
    # 距離ペナルティ追加
    distance_penalty = np.exp(-total_dist / 20)
    power *= distance_penalty
    
    return power

def find_best_combinations(self, top_n=10):
    """最適組合せ探索（高速版）"""
    print(f"\n最適組合せ探索中 (上位{top_n}組)...")
    self.update_status(stage='combination', progress=70, message='組合せ評価中')
    
    total = len(self.candidates['water_sources']) * \
            len(self.candidates['intakes']) * \
            len(self.candidates['powerhouses'])
    
    print(f"  評価対象: {total:,} 組合せ")
    
    results = []
    count = 0
    
    # 進捗バー
    from tqdm.notebook import tqdm
    with tqdm(total=total, desc="組合せ評価") as pbar:
        for ws in self.candidates['water_sources']:
            for intake in self.candidates['intakes']:
                for ph in self.candidates['powerhouses']:
                    power = self.estimate_power_generation(ws, intake, ph)
                    
                    if power > 0:
                        results.append({
                            'water_source': ws,
                            'intake': intake,
                            'powerhouse': ph,
                            'power_kw': power,
                            'head': ws['elevation'] - ph['elevation']
                        })
                    
                    count += 1
                    pbar.update(1)
                    
                    # 進捗更新（500組ごと）
                    if count % 500 == 0:
                        pct = int(70 + (count / total) * 20)  # 70%~90%
                        self.update_status(stage='combination', progress=pct,
                                         message=f'{count:,}/{total:,}組評価済')
    
    # 発電量順にソート
    results.sort(key=lambda x: x['power_kw'], reverse=True)
    self.best_combinations = results[:top_n]
    
    if self.best_combinations:
        print(f"✓ 上位{len(self.best_combinations)}組選定完了")
        print(f"  最高発電量: {self.best_combinations[0]['power_kw']:.1f} kW")
        print(f"  最大落差: {max(c['head'] for c in self.best_combinations):.1f} m")
    
    self.update_status(stage='combination', progress=90, message=f'上位{top_n}組選定完了')
    return self.best_combinations

# メソッド追加
HydroSiteSelector.estimate_power_generation = estimate_power_generation
HydroSiteSelector.find_best_combinations = find_best_combinations

print("✓ HydroSiteSelector クラス（パート3）定義完了")


In [ ]:
# ============================================================
# セル8: 可視化と分析実行（日本語出力版）
# ============================================================

def visualize_results(self):
    """結果の可視化（日本語メッセージ）"""
    print(f"\n結果を可視化中...")
    self.update_status(stage='visualize', progress=92, message='可視化処理中')
    
    # Matplotlibフォント設定（英語のみ使用）
    import matplotlib
    matplotlib.rcParams['font.family'] = 'DejaVu Sans'
    matplotlib.rcParams['axes.unicode_minus'] = False
    
    # 地図作成
    m = folium.Map(
        location=[self.center_lat, self.center_lon],
        zoom_start=10,
        tiles='OpenStreetMap'
    )
    
    # 標高の色マップ
    from matplotlib import cm
    from matplotlib.colors import Normalize
    
    norm = Normalize(vmin=self.elevation_data.min(), 
                    vmax=self.elevation_data.max())
    cmap = cm.terrain
    
    # グリッドポイントをサンプリング表示（高速化）
    sample_step = max(1, len(self.grid_points) // 200)  # 最大200点
    for i in range(0, len(self.grid_points), sample_step):
        lat, lon = self.grid_points[i]
        elev = self.elevation_data[i]
        color = cmap(norm(elev))
        color_hex = '#{:02x}{:02x}{:02x}'.format(
            int(color[0]*255), int(color[1]*255), int(color[2]*255))
        
        folium.CircleMarker(
            location=[lat, lon],
            radius=2,
            color=color_hex,
            fill=True,
            fillColor=color_hex,
            fillOpacity=0.5,
            popup=f"標高: {elev:.1f}m"
        ).add_to(m)
    
    # 上位組合せ表示（上位3組のみ）
    colors = ['red', 'blue', 'green']
    
    for i, combo in enumerate(self.best_combinations[:3]):  # 上位3組のみ
        color = colors[i]
        
        # 水源
        ws = combo['water_source']
        folium.Marker(
            location=[ws['lat'], ws['lon']],
            popup=f"<b>水源 #{i+1}</b><br>標高: {ws['elevation']:.1f}m<br>流量: {ws.get('estimated_flow', 0):.2f}m³/s",
            icon=folium.Icon(color=color, icon='tint', prefix='fa'),
            tooltip=f"水源 #{i+1}"
        ).add_to(m)
        
        # 取水口
        intake = combo['intake']
        folium.Marker(
            location=[intake['lat'], intake['lon']],
            popup=f"<b>取水口 #{i+1}</b><br>標高: {intake['elevation']:.1f}m",
            icon=folium.Icon(color=color, icon='filter', prefix='fa'),
            tooltip=f"取水口 #{i+1}"
        ).add_to(m)
        
        # 発電所
        ph = combo['powerhouse']
        folium.Marker(
            location=[ph['lat'], ph['lon']],
            popup=f"<b>発電所 #{i+1}</b><br>標高: {ph['elevation']:.1f}m<br>発電量: {combo['power_kw']:.1f}kW",
            icon=folium.Icon(color=color, icon='bolt', prefix='fa'),
            tooltip=f"発電所 #{i+1}"
        ).add_to(m)
        
        # ライン接続
        folium.PolyLine(
            locations=[
                [ws['lat'], ws['lon']],
                [intake['lat'], intake['lon']],
                [ph['lat'], ph['lon']]
            ],
            color=color,
            weight=3,
            opacity=0.7,
            popup=f"組合せ #{i+1}<br>発電量: {combo['power_kw']:.1f}kW"
        ).add_to(m)
    
    # 凡例（英語）
    legend_html = '''
    <div style="position: fixed; 
                top: 10px; right: 10px; width: 240px; height: auto; 
                background-color: white; z-index:9999; font-size:14px;
                border:2px solid grey; border-radius: 5px; padding: 10px">
    <p style="margin: 0; font-weight: bold;">Legend</p>
    <p style="margin: 5px 0;">💧 Water Source (High elevation)</p>
    <p style="margin: 5px 0;">🔵 Intake (Mid elevation)</p>
    <p style="margin: 5px 0;">⚡ Powerhouse (Low elevation)</p>
    </div>
    '''
    m.get_root().html.add_child(folium.Element(legend_html))
    
    print(f"✓ 地図生成完了")
    
    # グラフ作成
    graphs = {}
    
    # 1. 標高分布
    fig1, ax1 = plt.subplots(figsize=(10, 6))
    lats = [pt[0] for pt in self.grid_points]
    lons = [pt[1] for pt in self.grid_points]
    scatter = ax1.scatter(lons, lats, c=self.elevation_data, cmap='terrain', s=15, alpha=0.6)
    ax1.set_title('Elevation Distribution', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Longitude (deg)')
    ax1.set_ylabel('Latitude (deg)')
    plt.colorbar(scatter, ax=ax1, label='Elevation (m)')
    plt.tight_layout()
    graphs['elevation'] = fig1
    
    # 2. 発電量比較
    fig2, ax2 = plt.subplots(figsize=(10, 6))
    powers = [c['power_kw'] for c in self.best_combinations]
    ranks = [f'#{i+1}' for i in range(len(powers))]
    bars = ax2.bar(ranks, powers, color=['red', 'blue', 'green', 'purple', 'orange'][:len(powers)])
    ax2.set_title('Estimated Power Output Comparison', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Rank')
    ax2.set_ylabel('Power Output (kW)')
    ax2.grid(axis='y', alpha=0.3)
    
    # 値表示
    for bar in bars:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.0f}',
                ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    graphs['power'] = fig2
    
    # 3. 有効落差比較
    fig3, ax3 = plt.subplots(figsize=(10, 6))
    heads = [c['head'] for c in self.best_combinations]
    bars = ax3.bar(ranks, heads, color=['red', 'blue', 'green', 'purple', 'orange'][:len(heads)])
    ax3.set_title('Effective Head Comparison', fontsize=14, fontweight='bold')
    ax3.set_xlabel('Rank')
    ax3.set_ylabel('Effective Head (m)')
    ax3.grid(axis='y', alpha=0.3)
    
    # 値表示
    for bar in bars:
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.0f}',
                ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    graphs['head'] = fig3
    
    # 4. 標高プロファイル（上位3組）
    fig4, ax4 = plt.subplots(figsize=(10, 6))
    for i in range(min(3, len(self.best_combinations))):
        combo = self.best_combinations[i]
        elevs = [
            combo['water_source']['elevation'],
            combo['intake']['elevation'],
            combo['powerhouse']['elevation']
        ]
        positions = ['Water Source', 'Intake', 'Powerhouse']
        color = ['red', 'blue', 'green'][i]
        ax4.plot(positions, elevs, marker='o', linewidth=2, 
                markersize=10, label=f'Combination #{i+1}', color=color)
    
    ax4.set_title('Elevation Profile of Facilities', fontsize=14, fontweight='bold')
    ax4.set_ylabel('Elevation (m)')
    ax4.legend()
    ax4.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    graphs['profile'] = fig4
    
    print(f"✓ グラフ生成完了")
    
    return m, graphs

def run_analysis(self, grid_size=None, top_n=None, candidates_per_type=None):
    """分析実行（最適化版）"""
    print(f"\n{'='*60}")
    print(f"水力発電候補地選定システム")
    print(f"{'='*60}\n")
    self.update_status(stage='start', progress=1, message='分析開始')
    
    # 1. 座標取得
    if not self.get_location_coordinates():
        self.update_status(stage='error', progress=0, message='座標取得失敗')
        return None, None
    
    # 2. パラメータ自動調整
    if grid_size is None:
        target_density = 15  # km²/点（高速化のため粗く）
        total_points = max(400, min(10000, int(self.area_km2 / target_density)))
        grid_size = int(np.sqrt(total_points))
        print(f"\n📐 自動計算グリッドサイズ: {grid_size}x{grid_size} = {grid_size**2}点")
        print(f"   （目標密度: ~{target_density} km²/点, 面積: {self.area_km2:.1f} km²）")
    
    if candidates_per_type is None:
        total_grid = grid_size ** 2
        candidates_per_type = max(20, min(80, int(total_grid * 0.05)))
        print(f"📍 自動計算候補数/種類: {candidates_per_type}")
        print(f"   （グリッド点の5%, 総組合せ数: {candidates_per_type**3:,}）")
    
    if top_n is None:
        total_comb = candidates_per_type ** 3
        if total_comb < 10000:
            top_n = 10
        elif total_comb < 50000:
            top_n = 20
        else:
            top_n = 30
        print(f"🏆 自動計算出力数: 上位{top_n}組\n")
    
    # 3. グリッド生成
    self.generate_grid_points(grid_size=grid_size)
    
    # 4. 標高データ取得
    self.fetch_elevation_data(batch_size=150)  # バッチサイズ増（高速化）
    
    # 5. 河川データ取得
    self.fetch_river_data()
    
    # 6. 候補地選定（実際の河川データを使用）
    self.find_water_sources(top_n=candidates_per_type)
    self.find_intakes(top_n=candidates_per_type)
    self.find_powerhouses(top_n=candidates_per_type)
    
    # 7. 最適組合せ探索
    self.find_best_combinations(top_n=top_n)
    
    # 8. 可視化
    map_obj, fig = self.visualize_results()
    
    print(f"\n{'='*60}")
    print(f"分析完了!")
    print(f"{'='*60}\n")
    self.update_status(stage='done', progress=100, message='分析完了')
    
    return map_obj, fig

# メソッド追加
HydroSiteSelector.visualize_results = visualize_results
HydroSiteSelector.run_analysis = run_analysis

print("✓ HydroSiteSelector クラス（パート4）定義完了")


## 分析の実行

地名を指定して分析を実行します。指定した地名の**全範囲**を自動的に探索します。

※ 結果は自動的に `deta` フォルダに保存されます:
- トップ3のHTMLファイル
- トップ50のHTMLファイル
- CSVファイル
- 地図HTMLファイル

## 長野県での実行と評価

正確な行政区画境界を使用して、長野県の水力発電候補地を分析します。

In [ ]:
# ============================================================
# 長野県での水力発電候補地分析（ログ出力付き）
# ============================================================

import time

# ログ記録を開始
log_dir, log_path = start_logging("長野県")

try:
    print("="*70)
    print("長野県 - 水力発電候補地選定システム")
    print("="*70)

    # 分析開始
    start_time = time.time()

    # インスタンス作成
    nagano_selector = HydroSiteSelector("長野県")

    # パラメータ設定（自動計算）
    grid_size = 28        # 28x28 = 784点
    candidates = 40       # 各タイプ40候補
    top_n = 30           # 上位30組

    print(f"\n分析パラメータ:")
    print(f"  グリッドサイズ: {grid_size}x{grid_size} = {grid_size**2}点")
    print(f"  各候補数: {candidates}")
    print(f"  出力組合せ数: {top_n}")

    # 分析実行
    map_result, fig_result = nagano_selector.run_analysis(
        grid_size=grid_size,
        candidates_per_type=candidates,
        top_n=top_n
    )

    # 実行時間
    elapsed_time = time.time() - start_time

    print("="*70)
    print(f"分析完了! 実行時間: {elapsed_time:.1f}秒")
    print("="*70)
    
finally:
    # ログ記録を停止（必ず実行）
    stop_logging()
    
print(f"\n✓ すべての出力が以下のファイルに保存されました:")
print(f"  {log_path}")

In [ ]:
# ============================================================
# 結果の正当性評価（ログ出力付き）
# ============================================================

# ログ記録を開始（検証用）
log_dir, log_path = start_logging(f"長野県_validation")

try:
    print("="*70)
    print("出力データの正当性評価")
    print("="*70)

# 1. 地域範囲の検証
print("\n【1. 地域範囲の検証】")
print(f"地域名: {nagano_selector.location_name}")
print(f"境界頂点数: {len(nagano_selector.boundary_polygon.exterior.coords):,}")
print(f"面積: {nagano_selector.area_km2:.1f} km² (実際: 13,562 km²)")
print(f"面積誤差: {abs(nagano_selector.area_km2 - 13562) / 13562 * 100:.1f}%")

min_lat, max_lat, min_lon, max_lon = nagano_selector.bbox
print(f"\n緯度範囲: {min_lat:.4f}° ~ {max_lat:.4f}°")
print(f"経度範囲: {min_lon:.4f}° ~ {max_lon:.4f}°")

# 長野県の実際の範囲と比較
nagano_actual = {
    'lat': (35.2, 37.1),
    'lon': (137.3, 138.8),
    'elevation': (200, 3190),  # 天竜川沿い～奥穂高岳
    'avg_elevation': 1200
}

# 範囲チェック: 取得した範囲が長野県の実際の範囲内に収まっているか
range_ok = (nagano_actual['lat'][0] - 0.1 <= min_lat and
            max_lat <= nagano_actual['lat'][1] + 0.1 and
            nagano_actual['lon'][0] - 0.1 <= min_lon and
            max_lon <= nagano_actual['lon'][1] + 0.1)

print(f"\n範囲の妥当性: {'✓ 正常' if range_ok else '✗ 異常'}")

# 2. 標高データの検証
print("\n【2. 標高データの検証】")
elev_min = nagano_selector.elevation_data.min()
elev_max = nagano_selector.elevation_data.max()
elev_avg = nagano_selector.elevation_data.mean()
elev_median = np.median(nagano_selector.elevation_data)

print(f"標高範囲: {elev_min:.0f}m ~ {elev_max:.0f}m")
print(f"標高平均: {elev_avg:.0f}m (実際: 約{nagano_actual['avg_elevation']}m)")
print(f"標高中央値: {elev_median:.0f}m")

# 標高チェック: 長野県の実際の標高範囲に近いか
# 最低標高は200m~500m、最高標高は2500m~3190m、平均は1000m~1400m程度
elev_ok = (200 <= elev_min <= 500 and
           2500 <= elev_max <= 3200 and
           1000 <= elev_avg <= 1400)

# より詳細な評価を表示
elev_min_diff = abs(elev_min - nagano_actual['elevation'][0])
elev_max_diff = abs(elev_max - nagano_actual['elevation'][1])
elev_avg_diff = abs(elev_avg - nagano_actual['avg_elevation'])

print(f"  最低標高誤差: {elev_min_diff:.0f}m")
print(f"  最高標高誤差: {elev_max_diff:.0f}m")
print(f"  平均標高誤差: {elev_avg_diff:.0f}m (誤差率: {elev_avg_diff/nagano_actual['avg_elevation']*100:.1f}%)")

print(f"\n標高の妥当性: {'✓ 正常' if elev_ok else '✗ 異常'}")

# 3. 河川データの検証
print("\n【3. 河川データの検証】")
print(f"検出河川数: {len(nagano_selector.river_data):,}本")

river_types = {}
for river in nagano_selector.river_data:
    rtype = river['type']
    river_types[rtype] = river_types.get(rtype, 0) + 1

print(f"\n河川タイプ別:")
for rtype, count in river_types.items():
    print(f"  {rtype}: {count:,}本 ({count/len(nagano_selector.river_data)*100:.1f}%)")

river_ok = len(nagano_selector.river_data) > 10000  # 長野県は山岳地帯で河川が多い

print(f"\n河川数の妥当性: {'✓ 正常' if river_ok else '✗ 異常'}")

# 4. 候補地の検証
print("\n【4. 候補地の検証】")
ws_count = len(nagano_selector.candidates['water_sources'])
intake_count = len(nagano_selector.candidates['intakes'])
ph_count = len(nagano_selector.candidates['powerhouses'])

print(f"水源候補: {ws_count}箇所")
print(f"取水口候補: {intake_count}箇所")
print(f"発電所候補: {ph_count}箇所")
print(f"総組合せ数: {ws_count * intake_count * ph_count:,}")

# 水源の標高分布
ws_elevs = [ws['elevation'] for ws in nagano_selector.candidates['water_sources']]
print(f"\n水源の標高範囲: {min(ws_elevs):.0f}m ~ {max(ws_elevs):.0f}m")
print(f"水源の平均標高: {np.mean(ws_elevs):.0f}m")

# 発電所の標高分布
ph_elevs = [ph['elevation'] for ph in nagano_selector.candidates['powerhouses']]
print(f"\n発電所の標高範囲: {min(ph_elevs):.0f}m ~ {max(ph_elevs):.0f}m")
print(f"発電所の平均標高: {np.mean(ph_elevs):.0f}m")

candidates_ok = ws_count >= 30 and intake_count >= 30 and ph_count >= 30

print(f"\n候補地数の妥当性: {'✓ 正常' if candidates_ok else '✗ 異常'}")

# 5. 発電量の検証
print("\n【5. 発電量の検証】")
powers = [c['power_kw'] for c in nagano_selector.best_combinations]
heads = [c['head'] for c in nagano_selector.best_combinations]
flows = [c['water_source'].get('estimated_flow', 0) for c in nagano_selector.best_combinations]

print(f"最大発電量: {max(powers):.1f} kW")
print(f"最小発電量: {min(powers):.1f} kW")
print(f"平均発電量: {np.mean(powers):.1f} kW")

print(f"\n有効落差範囲: {min(heads):.0f}m ~ {max(heads):.0f}m")
print(f"平均有効落差: {np.mean(heads):.0f}m")

print(f"\n流量範囲: {min(flows):.2f} ~ {max(flows):.2f} m³/s")
print(f"平均流量: {np.mean(flows):.2f} m³/s")

# 小水力発電の一般的な範囲: 10kW ~ 1000kW
power_ok = 10 <= min(powers) <= 1000 and max(powers) <= 2000

print(f"\n発電量の妥当性: {'✓ 正常' if power_ok else '✗ 異常'}")

# 6. 上位3組の詳細
print("\n【6. 上位3組の詳細】")
for i, combo in enumerate(nagano_selector.best_combinations[:3], 1):
    ws = combo['water_source']
    intake = combo['intake']
    ph = combo['powerhouse']
    
    print(f"\n第{i}位: {combo['power_kw']:.1f} kW")
    print(f"  水源: ({ws['lat']:.4f}, {ws['lon']:.4f}) 標高{ws['elevation']:.0f}m 流量{ws.get('estimated_flow', 0):.2f}m³/s")
    print(f"  取水口: ({intake['lat']:.4f}, {intake['lon']:.4f}) 標高{intake['elevation']:.0f}m")
    print(f"  発電所: ({ph['lat']:.4f}, {ph['lon']:.4f}) 標高{ph['elevation']:.0f}m")
    print(f"  有効落差: {combo['head']:.0f}m")
    
    # 位置の妥当性チェック（長野県内か）
    from shapely.geometry import Point
    ws_in = nagano_selector.boundary_polygon.contains(Point(ws['lon'], ws['lat']))
    ph_in = nagano_selector.boundary_polygon.contains(Point(ph['lon'], ph['lat']))
    print(f"  位置検証: 水源{'✓' if ws_in else '✗'} 発電所{'✓' if ph_in else '✗'}")

# 総合評価
print("\n" + "="*70)
print("【総合評価】")
print("="*70)

all_checks = [range_ok, elev_ok, river_ok, candidates_ok, power_ok]
passed = sum(all_checks)
total = len(all_checks)

print(f"\n検証項目: {passed}/{total} 合格")

if passed == total:
    print("\n✓ すべての検証項目をクリアしました")
    print("✓ 出力データは妥当です")
elif passed >= total * 0.8:
    print(f"\n△ {total - passed}項目で問題がありますが、概ね妥当です")
else:
    print(f"\n✗ {total - passed}項目で問題があります。データを確認してください")

finally:
    # ログ記録を停止
    stop_logging()
    
print(f"\n✓ 検証結果が以下のファイルに保存されました:")
print(f"  {log_path}")

In [ ]:
# ============================================================
# 結果の保存（ログ出力付き）
# ============================================================

import os
from datetime import datetime
import pandas as pd

# ログ記録を開始（保存用）
log_dir, log_path = start_logging(f"長野県_save")

try:
    timestamp = datetime.now().strftime("%Y%m%d%H%M")
    output_dir = os.path.join("deta", timestamp)
    os.makedirs(output_dir, exist_ok=True)

    print("="*70)
    print("結果を保存中...")
    print("="*70)
    print(f"保存先: {output_dir}\n")

# 1. HTMLマップ保存（上位3組のみ）
map_filename = f"hydro_map_top3_{nagano_selector.location_name}_{timestamp}.html"
map_path = os.path.join(output_dir, map_filename)
map_result.save(map_path)
print(f"✓ 地図保存: {map_filename}")

# 2. CSVデータ保存
csv_data = []
for i, combo in enumerate(nagano_selector.best_combinations, 1):
    ws = combo['water_source']
    intake = combo['intake']
    ph = combo['powerhouse']
    
    from geopy.distance import geodesic
    ws_intake_dist = geodesic((ws['lat'], ws['lon']), (intake['lat'], intake['lon'])).kilometers
    intake_ph_dist = geodesic((intake['lat'], intake['lon']), (ph['lat'], ph['lon'])).kilometers
    
    csv_data.append({
        'Rank': i,
        'WaterSource_Lat': ws['lat'],
        'WaterSource_Lon': ws['lon'],
        'WaterSource_Elevation_m': ws['elevation'],
        'WaterSource_Flow_m3s': ws.get('estimated_flow', 0),
        'Intake_Lat': intake['lat'],
        'Intake_Lon': intake['lon'],
        'Intake_Elevation_m': intake['elevation'],
        'Powerhouse_Lat': ph['lat'],
        'Powerhouse_Lon': ph['lon'],
        'Powerhouse_Elevation_m': ph['elevation'],
        'EffectiveHead_m': combo['head'],
        'WS_Intake_Distance_km': ws_intake_dist,
        'Intake_PH_Distance_km': intake_ph_dist,
        'Total_Distance_km': ws_intake_dist + intake_ph_dist,
        'EstimatedPower_kW': combo['power_kw']
    })

df = pd.DataFrame(csv_data)
csv_filename = f"hydro_sites_{nagano_selector.location_name}_{timestamp}.csv"
csv_path = os.path.join(output_dir, csv_filename)
df.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f"✓ CSV保存: {csv_filename} ({len(df)}行)")

# 3. グラフ保存（英語版）
graph_names = {
    'elevation': '1_Elevation_Distribution',
    'power': '2_Power_Output_Comparison',
    'head': '3_Effective_Head_Comparison',
    'profile': '4_Facility_Elevation_Profile'
}

for key, name in graph_names.items():
    if key in fig_result:
        png_filename = f"{name}_{nagano_selector.location_name}_{timestamp}.png"
        png_path = os.path.join(output_dir, png_filename)
        fig_result[key].savefig(png_path, dpi=150, bbox_inches='tight')
        print(f"✓ グラフ保存: {png_filename}")

# 4. サマリー保存（英語版）
summary_filename = f"summary_{nagano_selector.location_name}_{timestamp}.txt"
summary_path = os.path.join(output_dir, summary_filename)

summary_lines = [
    "="*70,
    "Hydropower Site Selection System - Result Summary",
    "="*70,
    "",
    f"Region: {nagano_selector.location_name}",
    f"Date: {datetime.now().year}/{datetime.now().month}/{datetime.now().day} {datetime.now().strftime('%H:%M:%S')}",
    "",
    "[Basic Information]",
    f"Search Area: {nagano_selector.area_km2:.1f} km²",
    f"Grid Points: {len(nagano_selector.grid_points)}",
    f"Rivers Detected: {len(nagano_selector.river_data)}",
    f"Elevation Range: {nagano_selector.elevation_data.min():.1f}m ~ {nagano_selector.elevation_data.max():.1f}m",
    "",
    "[Candidate Sites]",
    f"Water Sources: {len(nagano_selector.candidates['water_sources'])}",
    f"Intakes: {len(nagano_selector.candidates['intakes'])}",
    f"Powerhouses: {len(nagano_selector.candidates['powerhouses'])}",
    "",
    "[Top 10 Power Output]"
]

for i, combo in enumerate(nagano_selector.best_combinations[:10], 1):
    ws = combo['water_source']
    summary_lines.append(
        f"#{i:2d}: {combo['power_kw']:6.1f} kW (Head: {combo['head']:6.1f}m, Flow: {ws.get('estimated_flow', 0):.2f}m³/s)"
    )

with open(summary_path, 'w', encoding='utf-8-sig') as f:
    f.write('\n'.join(summary_lines))

print(f"✓ サマリー保存: {summary_filename}")

print("\n" + "="*70)
print("保存完了!")
print("="*70)
print(f"\n保存ファイル:")
print(f"  {map_filename}")
print(f"  {csv_filename}")
print(f"  {summary_filename}")
for name in graph_names.values():
    print(f"  {name}_{nagano_selector.location_name}_{timestamp}.png")

finally:
    # ログ記録を停止
    stop_logging()
    
print(f"\n✓ 保存処理のログが以下のファイルに保存されました:")
print(f"  {log_path}")

In [ ]:
# ============================================================
# 修正したグラフの再生成と保存（ログ出力付き）
# ============================================================

# ログ記録を開始
log_dir, log_path = start_logging(f"長野県_regraph")

try:
    print("="*70)
    print("グラフを再生成して保存します...")
    print("="*70)

# 修正した可視化関数でグラフを再生成
map_result_fixed, fig_result_fixed = nagano_selector.visualize_results()

# 保存
import os
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d%H%M")
output_dir = f"deta/{timestamp}"
os.makedirs(output_dir, exist_ok=True)

location_name = nagano_selector.location_name

# グラフ保存（修正版）
graph_names = {
    'elevation': '1_Elevation_Distribution',
    'power': '2_Power_Output_Comparison',
    'head': '3_Effective_Head_Comparison',
    'profile': '4_Facility_Elevation_Profile'
}

for key, fig in fig_result_fixed.items():
    name = graph_names[key]
    png_filename = f"{name}_{location_name}_{timestamp}.png"
    png_path = os.path.join(output_dir, png_filename)
    fig.savefig(png_path, dpi=150, bbox_inches='tight')
    print(f"✓ グラフ保存: {png_filename}")

# 地図とCSVも保存
map_filename = f"hydro_map_top3_{location_name}_{timestamp}.html"
map_path = os.path.join(output_dir, map_filename)
map_result_fixed.save(map_path)
print(f"✓ 地図保存: {map_filename}")

# CSV保存
import pandas as pd
csv_data = []
for i, combo in enumerate(nagano_selector.best_combinations, 1):
    ws = combo['water_source']
    intake = combo['intake']
    ph = combo['powerhouse']
    csv_data.append({
        'Rank': i,
        'Power_kW': combo['power_kw'],
        'Effective_Head_m': combo['head'],
        'WS_Lat': ws['lat'],
        'WS_Lon': ws['lon'],
        'WS_Elevation_m': ws['elevation'],
        'WS_Flow_m3_s': ws.get('estimated_flow', 0),
        'Intake_Lat': intake['lat'],
        'Intake_Lon': intake['lon'],
        'Intake_Elevation_m': intake['elevation'],
        'PH_Lat': ph['lat'],
        'PH_Lon': ph['lon'],
        'PH_Elevation_m': ph['elevation'],
        'WS_Intake_Distance_km': combo.get('ws_intake_dist', 0),
        'Intake_PH_Distance_km': combo.get('intake_ph_dist', 0),
        'WS_PH_Distance_km': combo.get('ws_to_ph_dist', 0)
    })

df = pd.DataFrame(csv_data)
csv_filename = f"hydro_sites_{location_name}_{timestamp}.csv"
csv_path = os.path.join(output_dir, csv_filename)
df.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f"✓ CSV保存: {csv_filename} ({len(df)}行)")

print("\n" + "="*70)
print("修正版の保存完了!")
print("="*70)
print(f"\n保存先: {output_dir}")
print("\nファイル一覧:")
print(f"  {map_filename}")
print(f"  {csv_filename}")
for name in graph_names.values():
    print(f"  {name}_{location_name}_{timestamp}.png")

finally:
    # ログ記録を停止
    stop_logging()
    
print(f"\n✓ グラフ再生成のログが以下のファイルに保存されました:")
print(f"  {log_path}")